# Try CUGA

[CUGA](https://github.com/cuga-project/cuga-agent) is an open-source agent harness from IBM Research. Give it your tools and it plans and runs multi-step tasks over them, with policies that control what it may do.

This notebook installs CUGA, runs an agent with one tool, then adds a policy that blocks a request. You need an OpenAI API key; in Colab, add it under **Secrets** (🔑) as `OPENAI_API_KEY`.

## 1. Install

In [ ]:
%pip install -q cuga

## 2. Set your API key

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

## 3. Run an agent

Tools are plain Python functions. CUGA decides when to call them.

In [ ]:
from cuga import CugaAgent
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """Return the current weather for a city."""
    return f"Sunny and 24°C in {city}"


@tool
def cancel_trip(trip_id: str) -> str:
    """Cancel a booked trip."""
    return f"Trip {trip_id} cancelled"


agent = CugaAgent(tools=[get_weather, cancel_trip])

result = await agent.invoke("What's the weather in Tel Aviv?")
print(result.answer)

## 4. Add a policy

An intent guard stops matching requests before any tool runs.

In [ ]:
await agent.policies.add_intent_guard(
    name="No cancellations",
    description="Trips can only be cancelled by a travel admin",
    keywords=["cancel"],
    response="Cancelling trips isn't allowed here. Please contact a travel admin.",
    priority=100,
)

result = await agent.invoke("Cancel trip T-42")
print(result.answer)

## Next steps

- ⭐ [Star CUGA on GitHub](https://github.com/cuga-project/cuga-agent) to follow releases
- Run the full web UI with demo apps: [Quick Start](https://github.com/cuga-project/cuga-agent#quick-start)
- Other policy types (playbooks, tool approval, output formatting): [Policies guide](https://docs.cuga.dev/docs/sdk/policies/)